In [1]:
import os
import sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"  # 0 - first gpu, 1 - second, "0,1" - both gpu, first used, "-1" - none

os.environ['LD_LIBRARY_PATH'] = f"{sys.prefix}/lib:{os.environ.get('LD_LIBRARY_PATH', '')}"

import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tensorflow.keras.initializers import GlorotUniform, Orthogonal, Zeros
from tensorflow.keras.layers import ConvLSTM1D, Input, Reshape, Dropout, Flatten, Dense
from tensorflow.keras.models import Sequential, load_model

import os
import pickle
import numpy as np
import pandas as pd
from typing import List, Tuple, Dict, Any

BASE_PATH = "shared/explain-ts/ds/"
UNI_DIR = os.path.join(BASE_PATH, "univariate")
MULTI_DIR = os.path.join(BASE_PATH, "multivariate")

2026-03-03 20:57:00.870209: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
def get_dataset_names(path: str) -> List[str]:
    """
    Scans the specified directory and returns a list of dataset names.
    Ignores hidden files and directories.
    """
    return [f.name for f in os.scandir(path) if f.is_dir() and not f.name.startswith('.')]


def load_dataset_shapes(dataset_name: str, base_dir: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Loads strictly the data arrays from pickle files to avoid loading heavy models or explainers.
    """
    ds_path = os.path.join(base_dir, dataset_name)

    with open(os.path.join(ds_path, 'trainX.pickle'), 'rb') as f:
        trainX = pickle.load(f)
    with open(os.path.join(ds_path, 'trainy.pickle'), 'rb') as f:
        trainy = pickle.load(f)
    with open(os.path.join(ds_path, 'testX.pickle'), 'rb') as f:
        testX = pickle.load(f)
    with open(os.path.join(ds_path, 'testy.pickle'), 'rb') as f:
        testy = pickle.load(f)

    return trainX, trainy, testX, testy

In [3]:
def extract_descriptive_stats(
        trainX: np.ndarray,
        trainy: np.ndarray,
        testX: np.ndarray,
        dataset_name: str,
        ds_type: str
) -> Dict[str, Any]:
    """
    Extracts structural and descriptive statistics from standard time series arrays.
    """
    train_size, timesteps, channels = trainX.shape
    test_size = testX.shape[0]
    num_classes = trainy.shape[1]

    nan_train = np.isnan(trainX).sum()
    nan_test = np.isnan(testX).sum()

    return {
        "Dataset": dataset_name,
        "Type": ds_type,
        "Train_Size": train_size,
        "Test_Size": test_size,
        "Total_Size": train_size + test_size,
        "Timesteps": timesteps,
        "Channels": channels,
        "Classes": num_classes,
        "NaNs_Train": nan_train,
        "NaNs_Test": nan_test
    }

In [4]:
def generate_statistics_report(uni_dir: str, multi_dir: str) -> pd.DataFrame:
    """
    Orchestrates the extraction process across univariate and multivariate datasets.
    Returns a compiled pandas DataFrame with all statistics.
    """
    stats = []

    for ds in get_dataset_names(uni_dir):
        trainX, trainy, testX, _ = load_dataset_shapes(ds, uni_dir)
        ds_stats = extract_descriptive_stats(trainX, trainy, testX, ds, "Univariate")
        stats.append(ds_stats)

    for ds in get_dataset_names(multi_dir):
        trainX, trainy, testX, _ = load_dataset_shapes(ds, multi_dir)
        ds_stats = extract_descriptive_stats(trainX, trainy, testX, ds, "Multivariate")
        stats.append(ds_stats)

    return pd.DataFrame(stats)

In [5]:
# Execute extraction pipeline
df_stats = generate_statistics_report(UNI_DIR, MULTI_DIR)

In [14]:
df_stats

,Dataset,Type,Train_Size,Test_Size,Total_Size,Timesteps,Channels,Classes,NaNs_Train,NaNs_Test
0,DodgerLoopGame,Univariate,118,40,158,288,1,2,0,0
1,ProximalPhalanxTW,Univariate,453,152,605,80,1,6,0,0
2,DodgerLoopDay,Univariate,118,40,158,288,1,7,0,0
3,ECGFiveDays,Univariate,663,221,884,136,1,2,0,0
4,UMD,Univariate,135,45,180,150,1,3,0,0
...,...,...,...,...,...,...,...,...,...,...
99,Cricket,Multivariate,135,45,180,1197,6,12,0,0
100,LSST,Multivariate,3693,1232,4925,36,6,14,0,0
101,UWaveGestureLibrary,Multivariate,330,110,440,315,3,8,0,0
102,RacketSports,Multivariate,227,76,303,30,6,4,0,0


In [15]:
# Sort for better readability in the paper
df_stats.sort_values(by=["Type", "Total_Size"], ascending=[False, False], inplace=True)
df_stats = df_stats[df_stats["Dataset"] != "ChlorineConcentration"]

# Save to required formats
csv_path = os.path.join(BASE_PATH, "datasets_descriptive_statistics.csv")
latex_path = os.path.join(BASE_PATH, "datasets_descriptive_statistics.tex")

df_stats.to_csv(csv_path, index=False)
df_stats.to_latex(latex_path, index=False)

# Display the dataframe in Jupyter
df_stats

,Dataset,Type,Train_Size,Test_Size,Total_Size,Timesteps,Channels,Classes,NaNs_Train,NaNs_Test
37,Crop,Univariate,18000,6000,24000,46,1,24,0,0
24,ElectricDevices,Univariate,12477,4160,16637,96,1,7,0,0
53,Wafer,Univariate,5373,1791,7164,152,1,2,0,0
7,TwoPatterns,Univariate,3750,1250,5000,128,1,4,0,0
27,ECG5000,Univariate,3750,1250,5000,140,1,5,0,0
...,...,...,...,...,...,...,...,...,...,...
95,Epilepsy,Multivariate,206,69,275,206,3,4,0,0
87,HandMovementDirection,Multivariate,175,59,234,400,10,4,0,0
99,Cricket,Multivariate,135,45,180,1197,6,12,0,0
86,BasicMotions,Multivariate,60,20,80,100,6,4,0,0


In [6]:
def compute_benchmark_summary(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Computes summary statistics (min, median, max) for dataset characteristics
    to be used inline within the manuscript text.
    """
    summary = {
        "Total_Datasets": len(df),
        "Size_Min": df["Total_Size"].min(),
        "Size_Median": df["Total_Size"].median(),
        "Size_Max": df["Total_Size"].max(),
        "Length_Min": df["Timesteps"].min(),
        "Length_Median": df["Timesteps"].median(),
        "Length_Max": df["Timesteps"].max(),
        "Classes_Min": df["Classes"].min(),
        "Classes_Median": df["Classes"].median(),
        "Classes_Max": df["Classes"].max(),
        "Multi_Channels_Max": df[df["Type"] == "Multivariate"]["Channels"].max()
    }

    # Generate a draft sentence for the paper
    draft_text = (
        f"To provide a broad evaluation, the selected {summary['Total_Datasets']} datasets vary significantly in scale and complexity: "
        f"total instance counts range from {summary['Size_Min']} to {summary['Size_Max']} (median: {int(summary['Size_Median'])}), "
        f"time-series lengths span from {summary['Length_Min']} to {summary['Length_Max']} time steps (median: {int(summary['Length_Median'])}), "
        f"and the number of target classes varies between {summary['Classes_Min']} and {summary['Classes_Max']} (median: {int(summary['Classes_Median'])}). "
        f"While univariate datasets contain a single channel, the multivariate tasks include up to {summary['Multi_Channels_Max']} dimensions."
    )

    print("--- DRAFT FOR THE MANUSCRIPT ---")
    print(draft_text)

    return summary

In [17]:
df_stats.info()

<class 'pandas.core.frame.DataFrame'>
Index: 103 entries, 37 to 97
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Dataset     103 non-null    object
 1   Type        103 non-null    object
 2   Train_Size  103 non-null    int64 
 3   Test_Size   103 non-null    int64 
 4   Total_Size  103 non-null    int64 
 5   Timesteps   103 non-null    int64 
 6   Channels    103 non-null    int64 
 7   Classes     103 non-null    int64 
 8   NaNs_Train  103 non-null    int64 
 9   NaNs_Test   103 non-null    int64 
dtypes: int64(8), object(2)
memory usage: 12.9+ KB


In [18]:
df_stats

,Dataset,Type,Train_Size,Test_Size,Total_Size,Timesteps,Channels,Classes,NaNs_Train,NaNs_Test
37,Crop,Univariate,18000,6000,24000,46,1,24,0,0
24,ElectricDevices,Univariate,12477,4160,16637,96,1,7,0,0
53,Wafer,Univariate,5373,1791,7164,152,1,2,0,0
7,TwoPatterns,Univariate,3750,1250,5000,128,1,4,0,0
27,ECG5000,Univariate,3750,1250,5000,140,1,5,0,0
...,...,...,...,...,...,...,...,...,...,...
95,Epilepsy,Multivariate,206,69,275,206,3,4,0,0
87,HandMovementDirection,Multivariate,175,59,234,400,10,4,0,0
99,Cricket,Multivariate,135,45,180,1197,6,12,0,0
86,BasicMotions,Multivariate,60,20,80,100,6,4,0,0


In [19]:
stats_summary = compute_benchmark_summary(df_stats)

--- DRAFT FOR THE MANUSCRIPT ---
To provide a broad evaluation, the selected 103 datasets vary significantly in scale and complexity: total instance counts range from 30 to 24000 (median: 553), time-series lengths span from 8 to 1751 time steps (median: 235), and the number of target classes varies between 2 and 60 (median: 3). While univariate datasets contain a single channel, the multivariate tasks include up to 144 dimensions.


# Export all models Acc

In [7]:
# Load model

class SafeConvLSTM1D(ConvLSTM1D):
    def __init__(self, *args, **kwargs):
        kwargs.pop('time_major', None)
        super().__init__(*args, **kwargs)


class SafeGlorotUniform(tf.keras.initializers.GlorotUniform):
    def __init__(self, **kwargs):
        kwargs.pop('dtype', None)
        super().__init__(**kwargs)


class SafeOrthogonal(tf.keras.initializers.Orthogonal):
    def __init__(self, **kwargs):
        kwargs.pop('dtype', None)
        super().__init__(**kwargs)


class SafeZeros(tf.keras.initializers.Zeros):
    def __init__(self, **kwargs):
        kwargs.pop('dtype', None)  # Zeros might occasionally throw it too
        super().__init__()


# Crucial step: map the standard Keras names to our Safe wrappers
CUSTOM_OBJECTS = {
    'GlorotUniform': SafeGlorotUniform,
    'Orthogonal': SafeOrthogonal,
    'Zeros': SafeZeros,
    'ConvLSTM1D': SafeConvLSTM1D,
    'SafeConvLSTM1D': SafeConvLSTM1D
}


# --- Robust Loader ---
def load_benchmark_model(dataset_path: str, input_shape: tuple, num_classes: int) -> tf.keras.Model:
    h5_path = os.path.join(dataset_path, 'model.h5')
    tf_dir = os.path.join(dataset_path, 'model_tf/1')

    # 1. Standard load if healthy H5 exists
    if os.path.exists(h5_path):
        # We MUST pass CUSTOM_OBJECTS here to intercept 'dtype' during from_config()
        return load_model(h5_path, custom_objects=CUSTOM_OBJECTS, compile=False)

    # 2. Repair & Repack via Checkpoint Injection
    if os.path.isdir(tf_dir):
        print(f"Repacking legacy model for {os.path.basename(dataset_path)}...")

        # Build identical architecture using Safe layers to avoid initialization errors
        model = Sequential([
            Input(shape=input_shape),
            Reshape((*input_shape, 1), name='reshape'),
            SafeConvLSTM1D(64, kernel_size=3, padding='same', return_sequences=True, name='conv_lstm1d'),
            SafeConvLSTM1D(32, kernel_size=3, padding='same', return_sequences=True, name='conv_lstm1d_1'),
            Dropout(0.2, name='dropout'),
            Flatten(name='embedding'),
            Dense(100, activation='relu', name='dense'),
            Dense(num_classes, activation='softmax', name='dense_1')
        ])

        ckpt_prefix = os.path.join(tf_dir, 'variables', 'variables')

        try:
            checkpoint = tf.train.Checkpoint(model=model)
            checkpoint.restore(ckpt_prefix).expect_partial()
        except Exception as e:
            print(f"Checkpoint restore warning: {e}. Trying native Keras load_weights...")
            model.load_weights(ckpt_prefix)

        # Save healthy version for future runs
        model.save(h5_path)
        print("Successfully repacked to clean model.h5!")
        return model

    raise FileNotFoundError(f"No model artifacts found in {dataset_path}")


def extract_dataset_metrics(dataset_path: str, dataset_name: str, dataset_type: str) -> Dict[str, Any]:
    """
    Loads test artifacts and the corresponding pre-trained model to extract metrics.
    Gracefully handles missing model.h5 files by returning None for evaluation metrics.
    """
    with open(os.path.join(dataset_path, 'testX.pickle'), 'rb') as f:
        X_test = pickle.load(f)

    with open(os.path.join(dataset_path, 'testy.pickle'), 'rb') as f:
        y_test = pickle.load(f)

    # Always extract dataset structure info
    y_true_classes = np.argmax(y_test, axis=1) if len(y_test.shape) > 1 else y_test
    unique_classes = np.unique(y_true_classes)
    timesteps = X_test.shape[1]
    channels = X_test.shape[2] if len(X_test.shape) > 2 else 1

    input_dim = X_test.shape[1:]
    n_classes = y_test.shape[1] if len(y_test.shape) > 1 else len(np.unique(y_test))

    metrics = {
        "Dataset": dataset_name,
        "Type": dataset_type,
        "Classes": len(unique_classes),
        "Test_Samples": X_test.shape[0],
        "Timesteps": timesteps,
        "Channels": channels,
        "Test_Accuracy": None,
        "Test_F1_Weighted": None,
        "Test_Precision_Weighted": None,
        "Test_Recall_Weighted": None
    }

    # Attempt to evaluate the model
    try:
        # Automatically handles H5, SavedModel (model_tf/1) and legacy compatibility
        model = load_benchmark_model(dataset_path, input_shape=input_dim, num_classes=n_classes)
        y_pred_proba = model.predict(X_test, verbose=0)
        y_pred_classes = np.argmax(y_pred_proba, axis=1)

        metrics["Test_Accuracy"] = accuracy_score(y_true_classes, y_pred_classes)
        metrics["Test_F1_Weighted"] = f1_score(y_true_classes, y_pred_classes, average='weighted')
        metrics["Test_Precision_Weighted"] = precision_score(y_true_classes, y_pred_classes, average='weighted',
                                                             zero_division=0)
        metrics["Test_Recall_Weighted"] = recall_score(y_true_classes, y_pred_classes, average='weighted',
                                                       zero_division=0)

    except (FileNotFoundError, OSError):
        # Catch OSError as well, since h5py raises it when file is missing/corrupted
        print(f" [!] Missing or corrupted model for {dataset_name}. Storing structure only.")

    return metrics


def compile_all_model_evaluations(uni_dir: str, multi_dir: str) -> pd.DataFrame:
    """
    Iterates over all datasets to compile metadata and evaluation metrics.
    """
    results = []

    # Process Univariate
    for ds_name in [f.name for f in os.scandir(uni_dir) if f.is_dir() and not f.name.startswith('.')]:
        ds_path = os.path.join(uni_dir, ds_name)
        print(f"Evaluating {ds_name} (Univariate)...")
        results.append(extract_dataset_metrics(ds_path, ds_name, "Univariate"))

    # Process Multivariate
    for ds_name in [f.name for f in os.scandir(multi_dir) if f.is_dir() and not f.name.startswith('.')]:
        ds_path = os.path.join(multi_dir, ds_name)
        print(f"Evaluating {ds_name} (Multivariate)...")
        results.append(extract_dataset_metrics(ds_path, ds_name, "Multivariate"))

    return pd.DataFrame(results)


# Execute the extraction pipeline
df_evaluations = compile_all_model_evaluations(UNI_DIR, MULTI_DIR)

# Sort logically for presentation
df_evaluations.sort_values(by=["Type", "Dataset"], inplace=True)

Evaluating DodgerLoopGame (Univariate)...


/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
I0000 00:00:1772571441.749401 3824227 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6739 MB memory:  -> device: 0, name: NVIDIA RTX A5500, pci bus id: 0000:51:00.0, compute capability: 8.6
I0000 00:00:1772571441.749895 3824227 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 1961 MB memory:  -> device: 1, name: NVIDIA RTX A5500, pci bus id: 0000:9c:00.0, compute capability: 8.6
/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(

Evaluating ProximalPhalanxTW (Univariate)...


/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Evaluating DodgerLoopDay (Univariate)...


...

/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Evaluating RacketSports (Multivariate)...


/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
2026-03-03 21:01:58.159381: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-03-03 21:01:58.393349: I external/local_xla/xl

Evaluating SelfRegulationSCP1 (Multivariate)...


/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/home/jovyan/.conda/envs/explaints/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [8]:
missing_datasets = df_evaluations[df_evaluations["Test_Accuracy"].isna()]
display(missing_datasets[["Dataset", "Type", "Test_Samples", "Timesteps", "Channels"]])

,Dataset,Type,Test_Samples,Timesteps,Channels


In [10]:
df_evaluations

,Dataset,Type,Classes,Test_Samples,Timesteps,Channels,Test_Accuracy,Test_F1_Weighted,Test_Precision_Weighted,Test_Recall_Weighted
83,ArticularyWordRecognition,Multivariate,25,144,144,9,0.972222,0.973491,0.980035,0.972222
96,AtrialFibrillation,Multivariate,3,8,640,2,0.500000,0.412500,0.535714,0.500000
85,BasicMotions,Multivariate,4,20,100,6,1.000000,1.000000,1.000000,1.000000
98,Cricket,Multivariate,12,45,1197,6,0.911111,0.906936,0.931111,0.911111
91,ERing,Multivariate,6,75,65,4,0.960000,0.960112,0.967500,0.960000
...,...,...,...,...,...,...,...,...,...,...
57,Wine,Univariate,2,28,234,1,0.607143,0.458730,0.368622,0.607143
80,WordSynonyms,Univariate,25,227,270,1,0.678414,0.667364,0.694842,0.678414
39,Worms,Univariate,5,65,900,1,0.523077,0.526545,0.579878,0.523077
17,WormsTwoClass,Univariate,2,65,900,1,0.507692,0.507692,0.507692,0.507692


In [9]:
csv_output_path = os.path.join(BASE_PATH, "data_and_model_evaluations.csv")
df_evaluations.to_csv(csv_output_path, index=False)
print(f"\nSuccessfully saved evaluation results to: {csv_output_path}")


Successfully saved evaluation results to: shared/explain-ts/ds/data_and_model_evaluations.csv
